# 储备池计算（ESN）伪代码与可运行示例

本 notebook 包含：
1. ESN 核心实现的伪代码
2. 用正弦波做玩具数据验证 ESN 正确性
3. 加载 CPI 数据并做预测
4. 参数扫描模板

**注意**：这是第3阶段 `src/reservoir.py` 的骨架代码。正式实现时，把核心函数移到 `src/reservoir.py`，本 notebook 只保留调用和可视化。

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# 添加项目根目录到 path
sys.path.insert(0, str(Path.cwd().parent))

from src.config import DATA_PROCESSED_DIR, FIGURES_DIR, TABLES_DIR, RANDOM_SEED, DEFAULT_WINDOW_SIZE

np.random.seed(RANDOM_SEED)
%matplotlib inline

## 1. ESN 核心函数

下面实现 ESN 的四个核心函数：初始化储备池、收集状态、训练输出层、预测。

In [7]:
def init_reservoir(n_reservoir=200, spectral_radius=0.9, sparsity=0.05,
                   leak_rate=0.3, input_scaling=0.5, random_seed=42):
    """初始化 ESN 储备池参数
    
    参数:
        n_reservoir: 储备池节点数
        spectral_radius: 目标谱半径
        sparsity: 储备池连接的稀疏度
        leak_rate: 泄露率 α
        input_scaling: 输入缩放因子
        random_seed: 随机种子
    
    返回:
        dict: 包含 W_in, W, leak_rate, input_scaling
    """
    rng = np.random.RandomState(random_seed)
    
    # 输入权重矩阵 (n_reservoir x 1)，均匀分布
    W_in = rng.uniform(-1, 1, (n_reservoir, 1)) * input_scaling
    
    # 储备池内部连接矩阵，稀疏随机
    W = rng.uniform(-1, 1, (n_reservoir, n_reservoir))
    mask = rng.rand(n_reservoir, n_reservoir) < sparsity
    W = W * mask
    
    # 缩放使谱半径 = spectral_radius
    current_radius = max(abs(np.linalg.eigvals(W)))
    W = W * (spectral_radius / current_radius)
    
    return {
        'W_in': W_in,
        'W': W,
        'leak_rate': leak_rate,
        'input_scaling': input_scaling,
        'n_reservoir': n_reservoir
    }

In [8]:
def collect_states(X, reservoir, washout=10):
    """收集所有时间步的储备池状态
    
    参数:
        X: 输入序列，形状 (n_samples, n_features)
        reservoir: init_reservoir 返回的字典
        washout: 丢弃前 washout 步的瞬态
    
    返回:
        states: 储备池状态矩阵，形状 (n_samples - washout, n_reservoir)
    """
    W_in = reservoir['W_in']
    W = reservoir['W']
    alpha = reservoir['leak_rate']
    n_reservoir = reservoir['n_reservoir']
    n_samples = X.shape[0]
    
    # 初始化状态
    states = np.zeros((n_samples, n_reservoir))
    x = np.zeros(n_reservoir)
    
    for t in range(n_samples):
        u = X[t].reshape(-1, 1)  # 输入 (n_features, 1)
        # 状态更新：x(t) = (1-α)·x(t-1) + α·tanh(W_in·u(t) + W·x(t-1))
        x_new = (1 - alpha) * x + alpha * np.tanh(W_in @ u + W @ x).flatten()
        x = x_new
        states[t] = x
    
    # 丢弃前 washout 步
    return states[washout:]


def train_output(states, y, ridge_beta=1e-4):
    """用岭回归训练输出权重
    
    参数:
        states: 储备池状态，形状 (n_samples, n_reservoir)
        y: 目标值，形状 (n_samples,)
        ridge_beta: 正则化系数
    
    返回:
        W_out: 输出权重，形状 (n_reservoir,)
    """
    X = states
    n_reservoir = X.shape[1]
    
    # W_out = Y · X^T · (X · X^T + β·I)^(-1)
    # 用 solve 比 inv 更稳定
    A = X.T @ X + ridge_beta * np.eye(n_reservoir)
    b = X.T @ y
    W_out = np.linalg.solve(A, b)
    
    return W_out


def predict_esn(X, reservoir, W_out, washout=10):
    """用训练好的 ESN 做预测
    
    参数:
        X: 输入序列
        reservoir: 储备池参数字典
        W_out: 训练好的输出权重
        washout: 丢弃的瞬态步数
    
    返回:
        y_pred: 预测值，形状 (n_samples - washout,)
    """
    states = collect_states(X, reservoir, washout)
    y_pred = states @ W_out
    return y_pred

## 2. 玩具数据验证：正弦波预测

用最简单的正弦波验证 ESN 实现是否正确。输入过去 12 个点，预测下 1 个点。

In [9]:
# 生成正弦波数据
t = np.linspace(0, 20, 500)
series = np.sin(t) + 0.1 * np.random.randn(500)  # 加入少量噪声

# 构造滑动窗口
window_size = 12
X_list, y_list = [], []
for i in range(len(series) - window_size):
    X_list.append(series[i:i+window_size])
    y_list.append(series[i+window_size])

X = np.array(X_list)
y = np.array(y_list)

# 按时间顺序划分
n_train = int(0.7 * len(X))
n_val = int(0.15 * len(X))

X_train, y_train = X[:n_train], y[:n_train]
X_val, y_val = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = X[n_train+n_val:], y[n_train+n_val:]

print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}, 测试集: {X_test.shape}")

训练集: (341, 12), 验证集: (73, 12), 测试集: (74, 12)


In [10]:
# 初始化 ESN
reservoir = init_reservoir(n_reservoir=200, spectral_radius=0.9, sparsity=0.05,
                          leak_rate=0.3, input_scaling=0.5, random_seed=42)

# 收集训练状态
train_states = collect_states(X_train, reservoir, washout=10)
y_train_trimmed = y_train[10:]  # 对齐 washout

# 训练输出权重
W_out = train_output(train_states, y_train_trimmed, ridge_beta=1e-4)

# 预测
y_pred = predict_esn(X_test, reservoir, W_out, washout=10)
y_test_trimmed = y_test[10:]

# 计算误差
mse = np.mean((y_pred - y_test_trimmed) ** 2)
nrmse = np.sqrt(mse) / (np.max(y_test_trimmed) - np.min(y_test_trimmed))
print(f"测试集 NRMSE: {nrmse:.4f}")
print(f"测试集 MSE: {mse:.4f}")

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 12 is different from 1)

In [ ]:
# 可视化
plt.figure(figsize=(12, 4))
plt.plot(y_test_trimmed[:100], label='真实值', alpha=0.8)
plt.plot(y_pred[:100], label='ESN 预测', alpha=0.8)
plt.legend()
plt.title('正弦波预测：真实值 vs ESN 预测值')
plt.xlabel('时间步')
plt.ylabel('值')
plt.grid(alpha=0.3)
plt.show()

print(f"如果 NRMSE < 0.3，说明 ESN 实现正确。当前 NRMSE = {nrmse:.4f}")

## 3. CPI 数据预测

用 ESN 预测真实的 CPI 数据。

In [11]:
# 加载 CPI 滑动窗口数据
X_train = np.load(DATA_PROCESSED_DIR / 'X_train.npy')
y_train = np.load(DATA_PROCESSED_DIR / 'y_train.npy')
X_val = np.load(DATA_PROCESSED_DIR / 'X_val.npy')
y_val = np.load(DATA_PROCESSED_DIR / 'y_val.npy')
X_test = np.load(DATA_PROCESSED_DIR / 'X_test.npy')
y_test = np.load(DATA_PROCESSED_DIR / 'y_test.npy')

print(f"训练集: X{X_train.shape}, y{y_train.shape}")
print(f"验证集: X{X_val.shape}, y{y_val.shape}")
print(f"测试集: X{X_test.shape}, y{y_test.shape}")
print(f"CPI 范围: {y_train.min():.2f} ~ {y_train.max():.2f}")

训练集: X(212, 12), y(212,)
验证集: X(45, 12), y(45,)
测试集: X(47, 12), y(47,)
CPI 范围: 98.19 ~ 108.74


In [12]:
# 数据标准化（归一化到 [-1, 1]）
cpi_mean = y_train.mean()
cpi_std = y_train.std()

X_train_norm = (X_train - cpi_mean) / cpi_std
X_val_norm = (X_val - cpi_mean) / cpi_std
X_test_norm = (X_test - cpi_mean) / cpi_std
y_train_norm = (y_train - cpi_mean) / cpi_std
y_val_norm = (y_val - cpi_mean) / cpi_std
y_test_norm = (y_test - cpi_mean) / cpi_std

print(f"标准化后 CPI 范围: {y_train_norm.min():.2f} ~ {y_train_norm.max():.2f}")

标准化后 CPI 范围: -1.99 ~ 3.16


In [13]:
# 初始化 ESN（CPI 数据变化慢，用较小的泄露率）
reservoir = init_reservoir(
    n_reservoir=200,
    spectral_radius=0.9,
    sparsity=0.05,
    leak_rate=0.3,       # CPI 变化慢，小泄露率
    input_scaling=0.5,
    random_seed=42
)

# 训练
train_states = collect_states(X_train_norm, reservoir, washout=10)
y_train_trimmed = y_train_norm[10:]
W_out = train_output(train_states, y_train_trimmed, ridge_beta=1e-4)

# 验证集预测
y_val_pred = predict_esn(X_val_norm, reservoir, W_out, washout=10)
y_val_trimmed = y_val_norm[10:]

# 测试集预测
y_test_pred = predict_esn(X_test_norm, reservoir, W_out, washout=10)
y_test_trimmed = y_test_norm[10:]

# 反标准化
y_val_pred_orig = y_val_pred * cpi_std + cpi_mean
y_val_trimmed_orig = y_val_trimmed * cpi_std + cpi_mean
y_test_pred_orig = y_test_pred * cpi_std + cpi_mean
y_test_trimmed_orig = y_test_trimmed * cpi_std + cpi_mean

ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 12 is different from 1)

In [ ]:
# 计算误差指标
'''
sys.path.insert(0, str(Path.cwd().parent))
from src.metrics import mae, rmse, mape

print("=== ESN CPI 预测结果 ===")
print(f"验证集 MAE:  {mae(y_val_trimmed_orig, y_val_pred_orig):.4f}")
print(f"验证集 RMSE: {rmse(y_val_trimmed_orig, y_val_pred_orig):.4f}")
print(f"验证集 MAPE: {mape(y_val_trimmed_orig, y_val_pred_orig):.2f}%")
print()
print(f"测试集 MAE:  {mae(y_test_trimmed_orig, y_test_pred_orig):.4f}")
print(f"测试集 RMSE: {rmse(y_test_trimmed_orig, y_test_pred_orig):.4f}")
print(f"测试集 MAPE: {mape(y_test_trimmed_orig, y_test_pred_orig):.2f}%")
'''

In [ ]:
# 可视化
plt.figure(figsize=(14, 5))
plt.plot(y_test_trimmed_orig, 'b-', label='真实 CPI', linewidth=1.5)
plt.plot(y_test_pred_orig, 'r--', label='ESN 预测', linewidth=1.5)
plt.legend(fontsize=12)
plt.title('CPI 预测：真实值 vs ESN 预测值（测试集）', fontsize=14)
plt.xlabel('测试样本序号', fontsize=12)
plt.ylabel('CPI', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'esn_prediction.png', dpi=150)
plt.show()

## 4. 参数扫描模板

对关键参数进行网格搜索，找到最优组合。

In [ ]:
def esn_param_search(X_train, y_train, X_val, y_val, param_grid):
    """ESN 参数网格搜索
    
    参数:
        X_train, y_train: 训练数据
        X_val, y_val: 验证数据
        param_grid: 参数字典，如 {'n_reservoir': [100, 200], 'spectral_radius': [0.7, 0.9]}
    
    返回:
        results: 列表，每项为 (params, val_rmse)
    """
    from itertools import product
    
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    results = []
    
    for combo in product(*values):
        params = dict(zip(keys, combo))
        
        reservoir = init_reservoir(
            n_reservoir=params['n_reservoir'],
            spectral_radius=params['spectral_radius'],
            leak_rate=params['leak_rate'],
            input_scaling=params.get('input_scaling', 0.5),
            sparsity=params.get('sparsity', 0.05),
            random_seed=42
        )
        
        train_states = collect_states(X_train, reservoir, washout=10)
        y_train_trimmed = y_train[10:]
        W_out = train_output(train_states, y_train_trimmed)
        
        y_val_pred = predict_esn(X_val, reservoir, W_out, washout=10)
        y_val_trimmed = y_val[10:]
        
        val_rmse = np.sqrt(np.mean((y_val_pred - y_val_trimmed) ** 2))
        results.append((params, val_rmse))
        print(f"{params} -> RMSE: {val_rmse:.4f}")
    
    # 按 RMSE 排序
    results.sort(key=lambda x: x[1])
    return results

# 参数网格（可扩展）
param_grid = {
    'n_reservoir': [50, 100, 200],
    'spectral_radius': [0.5, 0.7, 0.9, 0.99],
    'leak_rate': [0.1, 0.3, 0.5, 0.7, 1.0],
}

# 运行参数搜索（注意：全网格有 3×4×5=60 个组合，可能较慢）
# 建议先跑一个子集
print("参数搜索共 {} 个组合".format(
    len(param_grid['n_reservoir']) * len(param_grid['spectral_radius']) * len(param_grid['leak_rate'])
))
# results = esn_param_search(X_train_norm, y_train_norm, X_val_norm, y_val_norm, param_grid)
# print(f"\n最优参数: {results[0][0]}")
# print(f"最优 RMSE: {results[0][1]:.4f}")

## 5. 伪代码：延时 RC（第4阶段）

这里给出延时 RC 的伪代码框架，第4阶段在 `src/optical_reservoir.py` 中实现。

```
# 伪代码：延时 RC (Delay-based Reservoir Computing)

def delay_rc_pipeline(input_series, params):
    # 参数
    tau = params['tau']          # 延迟时间
    N = params['n_virtual']      # 虚节点数量
    theta = tau / N              # 虚节点间隔
    eta = params['feedback_gain']  # 反馈增益
    gamma = params['input_gain']   # 输入增益
    
    # 生成随机掩码
    mask = np.random.choice([-1, 1], N)
    
    # 初始化状态
    x = np.zeros(N)  # 虚节点状态
    states = []
    
    for u in input_series:
        # 输入保持 + 掩码
        I = u * np.ones(N)
        J = mask * I
        
        # 非线性节点（Mackey-Glass 或 L-K 方程）
        for i in range(N):
            x[i] = nonlinear_node(x[i-1], J[i], eta, gamma)
        
        states.append(x.copy())
    
    # 训练输出权重（岭回归）
    states = np.array(states)
    W_out = ridge_regression(states, target)
    
    return W_out
```

## 6. 检查清单

在正式提交前，确认以下检查项：

- [ ] 正弦波玩具数据上 NRMSE < 0.3
- [ ] CPI 数据加载正确，形状与 `data_description.md` 一致
- [ ] 训练/验证/测试集按时间顺序划分，没有随机打乱
- [ ] washout 步数正确对齐，X 和 y 没有错位
- [ ] 没有未来数据泄漏（测试集数据从未出现在训练中）
- [ ] 预测图已保存到 `results/figures/esn_prediction.png`
- [ ] 误差结果已保存到 `results/tables/esn_results.csv`